# Agentic RAG

The pipelines of the previous notebooks run a fixed sequence of stages. This notebook follows
the book section *Agentic RAG* and hands control to the model instead. We declare three tools,
write the reason-act-observe loop of the book, and let the agent assemble the flow for the
book's two-part question about the 2025 physics laureates. Then we run the same loop on other
questions, cut it off with a step budget, and add the guardrails at its two edges.

**Learning goals:**
- Declare tools with a name, a description, and parameters, and see the model choose among them
- Implement the reason-act-observe loop with tool calling and an iteration budget
- Follow an agent's trajectory: intent detection, query transformation, search expansion, fallback
- See one loop serve different intents with a different number of steps
- Add input, tool-call, and output guardrails to the loop

**Prerequisites:** ch07-02 (the RAG pipeline and the `nobel-2025` knowledge base), ch07-03 (generator, synthesizer, citations)

## Setup

In [1]:
import os
import json
import re

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"

import logging
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

from shared.display import print_table, display_md
from shared.collections import load_collection
from ch07_chunking import contains
from ch07_rag import build_units, SparseIndex, DenseIndex, retrieve
from ch07_agent import Runtime, ToolCall, make_tools, tool_result

<div style="border-left: 4px solid #64748B; background: rgba(100, 116, 139, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#64748B; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Models, AWS, and recorded runs</strong><br>
The agent is Claude on AWS Bedrock with tool calling. Without AWS credentials, every agent run replays a run recorded earlier, model replies and tool results alike, so the notebook shows the same trajectories offline. The <code>web_search</code> tool calls the public Wikipedia search API and needs an internet connection in a live run.
</div>

The knowledge base is the `nobel-2025` collection with the retrieval chain of ch07-02: hybrid
BM25 and dense retrieval, fused and reranked.

In [2]:
collection = load_collection("nobel-2025")
parents, children = build_units(collection, parent_size=2000, child_size=400)

bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
sparse = SparseIndex(parents)
dense = DenseIndex(children, bi_encoder)

def search(question, k=5):
    """Top-k parents for a question: hybrid retrieval, then cross-encoder reranking."""
    return [p for p, _ in retrieve(question, sparse, dense, children, parents,
                                   cross_encoder, top_k=k)]

tools = make_tools(search, parents)
display_md(f"Knowledge base ready: **{len(parents)} passages**. Tools: {', '.join(tools)}.")

Knowledge base ready: **634 passages**. Tools: wikipedia_search, people_search, web_search.

## Declaring the Tools

The book gives the agent three tools: a Wikipedia knowledge base, a people search that returns
structured JSON, and a web search as fallback. A tool declaration has a name, a description,
and a schema of its parameters. The description is what the model reads when it decides which
tool to call, so it is part of the program.

In [3]:
TOOLS = [
    {"name": "wikipedia_search",
     "description": "Search a Wikipedia knowledge base about the Nobel Prizes and return "
                    "the text of the 3 best matching passages with their source.",
     "input_schema": {"type": "object",
                      "properties": {"query": {"type": "string"}},
                      "required": ["query"]}},
    {"name": "people_search",
     "description": "Look up a person in the people directory. Returns JSON with name, "
                    "affiliation, and position, or an error if there is no record.",
     "input_schema": {"type": "object",
                      "properties": {"name": {"type": "string"}},
                      "required": ["name"]}},
    {"name": "web_search",
     "description": "Search the web. Returns titles and short snippets. "
                    "Use only when the other tools return nothing.",
     "input_schema": {"type": "object",
                      "properties": {"query": {"type": "string"}},
                      "required": ["query"]}},
]

Behind each name is an ordinary function. `wikipedia_search` runs the hybrid retrieval and
returns three passages. `people_search` reads a small directory, like an HR or CRM system; it
holds records for recent physics laureates, but, as directories are, it is incomplete.
`web_search` asks the public Wikipedia search API. One direct call of each:

In [4]:
samples = [("wikipedia_search", {"query": "2025 Nobel Prize in Physics"}),
           ("people_search", {"name": "John Clarke"}),
           ("people_search", {"name": "John M. Martinis"})]
rows = []
for name, args in samples:
    result = json.dumps(tools[name](**args), ensure_ascii=False)
    rows.append([name, json.dumps(args), result[:150] + ("..." if len(result) > 150 else "")])
print_table(rows, ["Tool", "Arguments", "Result (start)"])

| Tool             | Arguments                                | Result (start)                                                                                                                                            |
|:-----------------|:-----------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------|
| wikipedia_search | {"query": "2025 Nobel Prize in Physics"} | [{"source": "2025 Nobel Prizes > Introduction", "url": "https://en.wikipedia.org/w/index.php?oldid=1379966130", "text": "The 2025 Nobel Prizes were aw... |
| people_search    | {"name": "John Clarke"}                  | {"name": "John Clarke", "born": 1942, "nationality": "United Kingdom", "affiliation": "University of California, Berkeley", "position": "Professor Eme... |
| people_search    | {"name": "John M. Martinis"}             | {"error": "no record for 'John M. Martinis' in the people directory"}                                                                                     |

The directory has no record for Martinis. Keep that in mind for the walk through the loop.

## The Agent Loop

The loop is the one of the book. `rt.llm` is the reason step: the model reads the conversation
and either answers or requests tool calls. `rt.call_tool` is the act step. The tool results go
back into the conversation as the observation. The growing message list is the agent's working
memory, and `max_steps` is the budget that ends a loop that would not stop by itself.

In [5]:
SYSTEM = (
    "You answer questions about the Nobel Prizes with the tools provided. Base every "
    "statement on tool results, and name the source of each fact in brackets, like "
    "[John Clarke (physicist) > Introduction] or [people_search]. If the tools do not give "
    "the answer, say so."
)

def run_agent(question, run, max_steps=8):
    """The reason-act-observe loop. Returns the answer and the runtime with its trace."""
    rt = Runtime(run, tools)
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):                                   # iteration budget
        reply = rt.llm(messages, TOOLS, system=SYSTEM)           # reason
        messages.append(reply.message)
        if not reply.tool_calls:                                 # no further action: done
            rt.save()
            return reply.text, rt
        results = [tool_result(call.id, rt.call_tool(call.name, call.arguments,
                                                     reply.text if i == 0 else ""))
                   for i, call in enumerate(reply.tool_calls)]   # act
        messages.append({"role": "user", "content": results})    # observe
    rt.save()
    return "I could not complete the answer within the step budget.", rt

One reason step can request several tool calls at once, and the model runs them in parallel.
A small helper shows the trajectory: one row per tool call, with the reason step it came from,
the model's reasoning text when it wrote one, the arguments, and the start of the observation.

In [6]:
def show_trace(rt):
    rows = [[t["step"], t["reason"][:90] or "-", t["tool"], json.dumps(t["arguments"]),
             t["result"][:90] + "..."] for t in rt.trace]
    print_table(rows, ["Step", "Reasoning", "Tool", "Arguments", "Observation (start)"])

def short(text, n=160):
    """First n characters of an answer, on one line."""
    text = " ".join(text.split())
    return text[:n] + ("..." if len(text) > n else "")

## Walking the Loop: the 2025 Physics Nobel

The book's question has two parts, and the second depends on the answer to the first:

In [7]:
QUESTION = ("Who won the 2025 Nobel Prize in Physics, and which universities are they "
            "affiliated with?")

answer, rt = run_agent(QUESTION, run="nobel")
show_trace(rt)
display_md(f"**{rt.calls} model calls, {len(rt.trace)} tool calls"
           f"{'' if rt.live else ' (recorded run)'}.**\n\n**Answer:**\n\n{answer}")

|   Step | Reasoning         | Tool             | Arguments                                                   | Observation (start)                                                                           |
|-------:|:------------------|:-----------------|:------------------------------------------------------------|:----------------------------------------------------------------------------------------------|
|      1 | -                 | wikipedia_search | {"query": "2025 Nobel Prize in Physics"}                    | [{"source": "2025 Nobel Prizes > Introduction", "url": "https://en.wikipedia.org/w/index.p... |
|      2 | Now affiliations. | people_search    | {"name": "John Clarke"}                                     | {"name": "John Clarke", "born": 1942, "nationality": "United Kingdom", "affiliation": "Uni... |
|      2 | -                 | people_search    | {"name": "Michel Devoret"}                                  | {"name": "Michel Devoret", "born": 1953, "nationality": "France, United States", "affiliat... |
|      2 | -                 | people_search    | {"name": "John M. Martinis"}                                | {"error": "no record for 'John M. Martinis' in the people directory"}...                      |
|      3 | -                 | people_search    | {"name": "John Martinis"}                                   | {"error": "no record for 'John Martinis' in the people directory"}...                         |
|      3 | -                 | wikipedia_search | {"query": "John Martinis physicist affiliation university"} | [{"source": "John M. Martinis > Introduction", "url": "https://en.wikipedia.org/w/index.ph... |

**4 model calls, 6 tool calls.**

**Answer:**

The 2025 Nobel Prize in Physics went to John Clarke, Michel Devoret and John M. Martinis. The prize was "for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit" [List of Nobel laureates in Physics > Laureates; 2025 Nobel Prizes > Prizes].

**Affiliations**
- **John Clarke:** University of California, Berkeley, where he is Professor Emeritus [people_search].
- **Michel Devoret:**
  - He is Professor of Physics at the University of California, Santa Barbara [people_search].
  - He is also Professor Emeritus of Applied Physics at Yale University [people_search].
  - He is Chief Scientist for Quantum Hardware at Google Quantum AI, which isn't a university [people_search].
- **John M. Martinis:**
  - He is Professor of Physics at the University of California, Santa Barbara [John M. Martinis > Introduction].
  - He has been on the UCSB faculty since 2004 [John M. Martinis > Career].
  - He earned his B.S. and Ph.D. at UC Berkeley, where Clarke was his doctoral advisor [John M. Martinis > Early life and education].
  - The people directory had no record for him, so his affiliation comes from Wikipedia alone.
  - Wikipedia says he joined the start-up Silicon Quantum Computing in 2020 [John M. Martinis > Career]. That isn't a university either.

In short, the universities are UC Berkeley (Clarke), UC Santa Barbara (Devoret and Martinis), and Yale (Devoret, emeritus).

Read the trajectory against the five steps of the book:

1. **Intent detection.** No classifier ran. The agent's first action already goes after the
   first part, the winners, and its next step turns to the second part, the affiliations.
2. **Query transformation.** The first `wikipedia_search` does not pass the user's sentence;
   the agent writes its own, shorter query.
3. **Search expansion.** Once the names are known, the agent loops and looks up each laureate,
   typically with one parallel call per name in a single step. Nobody coded this per-winner
   chain; the agent assembled it from the question.
4. **Fallback.** The directory has no record for Martinis, so the agent has to get his
   affiliation another way: with a different spelling, from the knowledge base, or, as the
   last resort its description names, from the web.
5. **Final answer.** When the evidence is complete, the agent stops calling tools and answers,
   with the source of each fact in brackets.

The table below checks steps 2 and 4 on the trace itself:

In [8]:
first = rt.trace[0]
not_found = [i for i, t in enumerate(rt.trace) if '"error"' in t["result"]]
after = [rt.trace[i + 1] for i in not_found if i + 1 < len(rt.trace)]
print_table([
    ["User question", QUESTION],
    ["First tool call", f"{first['tool']}({json.dumps(first['arguments'])})"],
    ["Lookups that found nothing",
     "; ".join(f"{rt.trace[i]['tool']}({json.dumps(rt.trace[i]['arguments'])})" for i in not_found)
     or "none"],
    ["Next call after a miss",
     "; ".join(f"{t['tool']}({json.dumps(t['arguments'])})" for t in after) or "none"],
], ["", "Trajectory"])

|                            | Trajectory                                                                                                              |
|:---------------------------|:------------------------------------------------------------------------------------------------------------------------|
| User question              | Who won the 2025 Nobel Prize in Physics, and which universities are they affiliated with?                               |
| First tool call            | wikipedia_search({"query": "2025 Nobel Prize in Physics"})                                                              |
| Lookups that found nothing | people_search({"name": "John M. Martinis"}); people_search({"name": "John Martinis"})                                   |
| Next call after a miss     | people_search({"name": "John Martinis"}); wikipedia_search({"query": "John Martinis physicist affiliation university"}) |

The per-winner chain and the fallback are exactly what the fixed pipeline of ch07-02 could not
express, and what ch07-03 had to hard-code as a synthesizer plan. Here the plan came from the
question and from what each observation returned.

## One Loop, Many Intents

The same function, unchanged, answers questions of a different shape. A definition needs one
search; a count needs the right passage; the two-part question needs a chain.

In [9]:
runs = [("What is a transmon?", "transmon"),
        ("How many people have won the Nobel Prize in Physics as of 2026?", "count")]
rows = []
for q, run in runs:
    a, r = run_agent(q, run=run)
    rows.append([q, r.calls, len(r.trace), ", ".join(dict.fromkeys(t["tool"] for t in r.trace)) or "-",
                 short(a)])
rows.append([QUESTION, rt.calls, len(rt.trace),
             ", ".join(dict.fromkeys(t["tool"] for t in rt.trace)), "(above)"])
print_table(rows, ["Question", "Model calls", "Tool calls", "Tools", "Answer (start)"])

| Question                                                                                  |   Model calls |   Tool calls | Tools                           | Answer (start)                                                                                                                                                      |
|:------------------------------------------------------------------------------------------|--------------:|-------------:|:--------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| What is a transmon?                                                                       |             2 |            1 | wikipedia_search                | A transmon is a type of superconducting charge qubit used in superconducting quantum computing. It is designed to be less sensitive to charge noise [Transmon > ... |
| How many people have won the Nobel Prize in Physics as of 2026?                           |             2 |            2 | wikipedia_search                | 230 people have won the Nobel Prize in Physics as of 2026 [List of Nobel laureates in Physics > Statistics]. - John Bardeen won twice, in 1956 and 1972. The 230... |
| Who won the 2025 Nobel Prize in Physics, and which universities are they affiliated with? |             4 |            6 | wikipedia_search, people_search | (above)                                                                                                                                                             |

The number of steps and the tools used are decided by the question, not by the code. This is
the book's point: a single generic loop, given the right tools, replaces one pipeline per
intent.

## The Step Budget

The budget is the stop condition that does not depend on the model. With two steps, the agent
has no room to finish the chain:

In [10]:
answer_short, rt_short = run_agent(QUESTION, run="budget", max_steps=2)
show_trace(rt_short)
display_md(f"**Answer after {rt_short.calls} model calls:** {answer_short}")

|   Step | Reasoning         | Tool             | Arguments                                | Observation (start)                                                                           |
|-------:|:------------------|:-----------------|:-----------------------------------------|:----------------------------------------------------------------------------------------------|
|      1 | -                 | wikipedia_search | {"query": "2025 Nobel Prize in Physics"} | [{"source": "2025 Nobel Prizes > Introduction", "url": "https://en.wikipedia.org/w/index.p... |
|      2 | Now affiliations. | people_search    | {"name": "John Clarke"}                  | {"name": "John Clarke", "born": 1942, "nationality": "United Kingdom", "affiliation": "Uni... |
|      2 | -                 | people_search    | {"name": "Michel Devoret"}               | {"name": "Michel Devoret", "born": 1953, "nationality": "France, United States", "affiliat... |
|      2 | -                 | people_search    | {"name": "John M. Martinis"}             | {"error": "no record for 'John M. Martinis' in the people directory"}...                      |

**Answer after 2 model calls:** I could not complete the answer within the step budget.

A production agent sets the budget well above the steps a normal question needs, so it ends
only loops that have gone wrong: a tool that keeps failing, or a model that keeps asking the
same thing.

## Guardrails

The book places guardrails at the edges of the loop: an input check before the agent acts, and
an output check before the answer reaches the user. In an agent the model acts several times,
so the tool-call check runs before *every* action.

### Input: is the question in scope?

A short classification call decides whether the assistant should handle the question at all.
An out-of-scope question is declined before any tool runs.

In [11]:
SCOPE = ("You guard an assistant that answers questions about the Nobel Prizes, their "
         "laureates, and the laureates' research. Reply with IN if the question below is in "
         "scope and OUT otherwise, and nothing else.\n\nQuestion: {question}")

guard = Runtime("guards", tools)
rows = []
for q in [QUESTION, "Which pizza place in Basel has the best reviews?"]:
    verdict = guard.ask(SCOPE.format(question=q)).strip().upper()
    rows.append([q, verdict, "run the agent" if verdict.startswith("IN") else
                 "decline: I can only answer questions about the Nobel Prizes."])
print_table(rows, ["Question", "Scope check", "Action"])

| Question                                                                                  | Scope check   | Action                                                       |
|:------------------------------------------------------------------------------------------|:--------------|:-------------------------------------------------------------|
| Who won the 2025 Nobel Prize in Physics, and which universities are they affiliated with? | IN            | run the agent                                                |
| Which pizza place in Basel has the best reviews?                                          | OUT           | decline: I can only answer questions about the Nobel Prizes. |

### Tool calls: is the action allowed?

Before `call_tool` runs, the loop checks the request: only declared tools, only declared
arguments, no empty values. This check needs no model; it is plain code.

In [12]:
DECLARED = {t["name"]: set(t["input_schema"]["properties"]) for t in TOOLS}

def check_call(call):
    """Return None if the tool call is allowed, else the reason to block it."""
    if call.name not in DECLARED:
        return f"unknown tool {call.name!r}"
    if set(call.arguments) != DECLARED[call.name]:
        return f"unexpected arguments {sorted(call.arguments)}"
    if not all(str(v).strip() for v in call.arguments.values()):
        return "empty argument"
    return None

tests = [ToolCall("1", "people_search", {"name": "Michel Devoret"}),
         ToolCall("2", "delete_index", {"name": "nobel-2025"}),
         ToolCall("3", "web_search", {"query": " "})]
print_table([[c.name, json.dumps(c.arguments), check_call(c) or "allowed"] for c in tests],
            ["Tool", "Arguments", "Check"])

| Tool          | Arguments                  | Check                       |
|:--------------|:---------------------------|:----------------------------|
| people_search | {"name": "Michel Devoret"} | allowed                     |
| delete_index  | {"name": "nobel-2025"}     | unknown tool 'delete_index' |
| web_search    | {"query": " "}             | empty argument              |

To guard the loop, the act step runs `call_tool` only when `check_call` returns `None`;
otherwise the reason it was blocked becomes the observation, so the model sees the refusal and
can choose another action.

### Output: is every claim supported?

Before the answer is returned, a second model call compares it with the observations the agent
collected and lists any claim they do not support. We run it on the agent's answer, and on a
copy with one invented sentence added.

In [13]:
GROUNDING = (
    "Below are tool results and an answer. List every claim in the answer that the tool "
    'results do not support. Reply with JSON only: {{"supported": true or false, '
    '"unsupported_claims": ["..."]}}.\n\n<tool_results>\n{evidence}\n</tool_results>\n\n'
    "<answer>\n{answer}\n</answer>"
)
evidence = "\n\n".join(t["result"] for t in rt.trace)

def grounded(answer):
    reply = guard.ask(GROUNDING.format(evidence=evidence, answer=answer))
    return json.loads(re.search(r"\{.*\}", reply, re.S).group(0))

tampered = answer + "\n\nJohn M. Martinis also holds a chair at the University of Oxford."
rows = []
for name, text in [("Agent's answer", answer), ("With an invented sentence", tampered)]:
    check = grounded(text)
    rows.append([name, "yes" if check["supported"] else "**no**",
                 "; ".join(check["unsupported_claims"]) or "-"])
guard.save()
print_table(rows, ["Answer", "Supported?", "Unsupported claims"])

| Answer                    | Supported?   | Unsupported claims                                               |
|:--------------------------|:-------------|:-----------------------------------------------------------------|
| Agent's answer            | yes          | -                                                                |
| With an invented sentence | **no**       | John M. Martinis also holds a chair at the University of Oxford. |

The invented chair at Oxford reads as smoothly as the rest of the answer; only the comparison
with the evidence exposes it. When the check fails, the safe response is to drop the claim or to
answer with the uncertainty, never to pass the confident sentence on.

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Declare the capabilities, let the loop decide the flow</strong><br>
We did not write a winners-then-affiliations pipeline. We declared three tools, wrote a loop of a dozen lines, and the agent assembled the chain, including the fallback when the directory had no record. Guardrails take the place of the fixed sequence: a scope check before the loop, a check before every action, and a grounding check before the answer.
</div>

## Summary

| Element | Here | Role |
|---|---|---|
| Tool declaration | name, description, input schema | what the agent may do; the description guides the choice |
| Reason | one `invoke_claude_tools` turn | answer, or request tool calls |
| Act | `call_tool` | run a tool with the model's arguments |
| Observe | `tool_result` messages | results become the agent's working memory |
| Step budget | `max_steps` | stops a loop that does not end by itself |
| Guardrails | scope check, `check_call`, grounding check | keep each decision in scope and each claim supported |

The retrieval chain of ch07-02 is still here, but as one tool among three. The agent supplies
the control flow; we supply the tools, the data behind them, and the checks.

## Try it yourself

1. Add a record for John M. Martinis to `PEOPLE` in <a href="ch07_agent.py">ch07_agent.py</a>
   and run the Nobel question again. How many tool calls does the agent need now?
2. Change the description of `web_search` to "Search the web for any question." Does the agent
   now prefer it over the knowledge base?
3. Ask a question that needs a comparison, such as "Which 2024 or 2025 physics laureate is the
   oldest?" Which tools does the agent call, and in what order?